In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math 

In [ ]:
df = pd.read_csv("datasets/raw-data/nilai-gizi-fitur.csv", sep=";")
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
#Data Cleaning dan Pre-processing EDA = utk mengetahui penyakit data
df.isnull().sum()

In [ ]:
df['name'] #data baru yg telah diproses

In [ ]:
# deteksi outliers dgn seaborn
cols = ['energy_kcal', 'protein_g', 'carbohydrate_g', 'fat_g', 'sugar_g', 'sodium_mg', 'fiber_g']

sns.boxplot(data=df[cols])

In [ ]:
print(df[cols].dtypes) # cek sebelumnya masih ada type object

In [ ]:
cols_fix = ['energy_kcal', 'protein_g', 'fat_g', 'carbohydrate_g']

for col in cols_fix:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .str.replace(' g', '', regex=False)
        .str.replace(' mg', '', regex=False)
        .replace(['Tidak Diketahui', 'tidak diketahui', 'TIDAK DIKETAHUI'], pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
print(df[cols].dtypes) # cek tipe data

In [ ]:
# hapus baris data yg ga lengkap
FEATURES = [
    'sugar_g',
    'sodium_mg',
    'fat_g',
    'energy_kcal',
    'protein_g',
    'fiber_g',
    'carbohydrate_g'
]

df = df.dropna(subset=FEATURES)

In [ ]:
# hapus kadar nutrisi yang mustahil
df = df[
    (df['protein_g'] <= 90) &
    (df['fat_g'] <= 80) &
    (df['carbohydrate_g'] <= 400) &
    (df['sugar_g'] <= 50) &
    (df['fiber_g'] <= 40) &
    (df['energy_kcal'] <= 2700)
]

In [ ]:
# simpan hasil logic threshold
df = df.copy()
sns.boxplot(data=df[FEATURES])

In [ ]:
df[FEATURES].describe(percentiles=[0.5, 0.75, 0.9]) # ambil nilai yang 75%

In [ ]:
# FEATURE ENGINEERING: NUTRITION SCORE


limits = df[[
    'energy_kcal',
    'sugar_g',
    'fat_g',
    'sodium_mg',
    'carbohydrate_g'
]].quantile(0.60) # diturunin dari 0.75

bonus = df[[
    'protein_g',
    'fiber_g'
]].quantile(0.50)

def hitung_skor(row):
    skor = 0.0

    # poin negatif berbobot (SAMA PERSIS dengan klasifikasi_nutrisi)
    if row['sugar_g'] >= limits['sugar_g']:
        skor += 2.5
    if row['sodium_mg'] >= limits['sodium_mg']:
        skor += 2.5
    if row['fat_g'] >= limits['fat_g']:
        skor += 1.5
    if row['energy_kcal'] >= limits['energy_kcal']:
        skor += 1.0
    if row['carbohydrate_g'] >= limits['carbohydrate_g']:
        skor += 0.5

    # poin positif (maks 1)
    bonus_count = 0
    if row['protein_g'] >= bonus['protein_g']:
        bonus_count += 1
    if row['fiber_g'] >= bonus['fiber_g']:
        bonus_count += 1

    skor -= min(bonus_count, 1)

    return skor


# Tambahkan skor ke dataset
df['nutrition_score'] = df.apply(hitung_skor, axis=1)


In [ ]:
# ambang batas
q45 = df['nutrition_score'].quantile(0.45)
q75 = df['nutrition_score'].quantile(0.75)

def klasifikasi_nutrisi(row):

    if row['nutrition_score'] <= q45:
        return 'Sehat'
    elif row['nutrition_score'] <= q75:
        return "Kurang Sehat"
    else:
        return 'Tidak Sehat'

In [ ]:
df['label'] = df.apply(klasifikasi_nutrisi, axis=1) # simpan label dalam setiap baris dataset

# untuk sanity check
df[['nutrition_score', 'label']].head()

In [ ]:
# Cek banyak label
print(df['label'].value_counts())


In [ ]:
# cek sample label
df[['name', 'energy_kcal', 'sugar_g', 'fat_g', 'sodium_mg', 'protein_g', 'fiber_g', 'label']] \
    .sample(10)

In [ ]:
# simpan logic dan buat dataset bersih dalam bentuk .csv
df_final = df.copy()
df_final.to_csv("datasets/processed/nutrisi-cleaned.csv", index=False)

In [ ]:
# Load dataset dan persiapan
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("datasets/processed/nutrisi-cleaned.csv") #load dataset yang udh diproses

# Nentuin Fitur(X) dan Target(Y)
fitur_gizi = [
    'sugar_g',
    'sodium_mg',
    'fat_g',
    'energy_kcal',
    'protein_g',
    'fiber_g',
    'carbohydrate_g'
]

X = df[fitur_gizi]
y = df['label'] 

# encoding label
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("\nMapping Label:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

# train test split stratified

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Distribusi label:")
print(df['label'].value_counts(normalize=True)) # tadinya gada isi
print("\nMapping label:", dict(zip(le.classes_, le.transform(le.classes_))))
print("\nJumlah data train:", len(X_train))
print("Jumlah data test :", len(X_test))

In [ ]:
print("X_train shape:", X_train.shape)

In [ ]:
# class imbalance handling (buat rf dan xgb)
from sklearn.utils.class_weight import compute_class_weight

# hitung bobot kelas
classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes = classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print("Class Weights:", class_weight_dict)

In [ ]:
# Scaling dan Cross-Validation
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# scaling khusus untuk KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# pendefinisian model
models = [
    ('Random Forest', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced'   
    )),
    ('XGBoost', XGBClassifier(
        eval_metric='mlogloss',
        random_state=42
    )),
    ('KNN', KNeighborsClassifier(n_neighbors=5))
]

# Stratifiend Cross-Val
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = 'f1_macro'

print("=== CROSS VALIDATION (F1 MACRO) ===")

for name, model in models:
    if name == 'KNN':
        scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring=scoring)
    else:
        scores = cross_val_score(model, X_train, y_train, cv=skf, scoring=scoring)

    print(f"{name}: Mean = {scores.mean():.4f}, Std = {scores.std():.4f}")

In [ ]:
# Random Forest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

#definisi target name dari LabelEncoder
target_names = le.classes_

#training rf
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    # class_weight='balanced'
    class_weight=class_weight_dict
)

rf_model.fit(X_train, y_train)

#prediksi
y_pred_rf = rf_model.predict(X_test)

#evaluasi
print("=== Evaluasi RF ===")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=target_names
))

# confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_rf,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=target_names,
    yticklabels=target_names
)
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

In [ ]:
#XGBoost
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    
    max_depth=4, # perubahan
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Prediksi
y_pred_xgb = xgb_model.predict(X_test)

#evaluasi
print("=== EVALUASI XGBOOST ===")
print(classification_report(
    y_test,
    y_pred_xgb,
    # target_names=target_names
    target_names=['Sehat', 'Kurang Sehat', 'Tidak Sehat']
))

# confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_xgb,
    annot=True,
    fmt='d',
    cmap='Oranges',
    xticklabels=target_names,
    yticklabels=target_names
)
plt.title("Confusion Matrix - XGBoost")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8,5))
plt.bar(range(len(importances)), importances[indices], color='orange')
plt.xticks(range(len(importances)), np.array(X.columns)[indices], rotation=45)
plt.title("Feature Importance - XGBoost")
plt.tight_layout()
plt.show()

In [ ]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# scaling data test (WAJIB)
X_test_scaled = scaler.transform(X_test)

# training
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

# prediksi
y_pred_knn = knn_model.predict(X_test_scaled)

# evaluasi
print("=== EVALUASI KNN ===")
print(classification_report(
    y_test,
    y_pred_knn,
    target_names=target_names
))

# confusion matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_knn,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=target_names,
    yticklabels=target_names
)
plt.title("Confusion Matrix - KNN")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

In [ ]:
import joblib

paket_model = {
    'model': xgb_model,          # ← XGBOOST
    'scaler': scaler,            # scaler yang sama dari training
    'label_encoder': le,         # encoder label
    'features': fitur_gizi       # urutan fitur FINAL
}

joblib.dump(paket_model, "model/model-train.pkl")
print("✅ Model FINAL (XGBoost) berhasil disimpan")

In [ ]:
paket = joblib.load("model/model-train.pkl")

model = paket['model']
scaler = paket['scaler']
le = paket['label_encoder']
features = paket['features']

In [ ]:
# test data baru masuk buat prediksi

# sehat
# data_baru = [[
#     5,     # sugar_g
#     300,   # sodium_mg
#     6,     # fat_g
#     180,   # energy_kcal
#     8,     # protein_g
#     3,     # fiber_g
#     30     # carbohydrate_g
# ]]

# kurang sehat
# data_baru = [[
#     36,
#     1000,
#     18,
#     420,  
#     1,
#     0,
#     54
# ]]

#tidak sehat
data_baru = [[
    50,    # sugar_g
    3000,  # sodium_mg
    100,    # fat_g
    2300,   # energy_kcal
    0,     # protein_g
    0,     # fiber_g
    1000     # carbohydrate_g
]]

df_baru = pd.DataFrame(data_baru, columns=fitur_gizi)

In [ ]:
data_scaled = scaler.transform(df_baru) #scaling 

In [ ]:
# hasil prediksi data baru
pred_angka = model.predict(data_scaled)[0]
pred_label = le.inverse_transform([pred_angka])[0]

print("Hasil klasifikasi:", pred_label)

In [ ]:
# probabilitas label
proba = model.predict_proba(data_scaled)[0]

for lbl, p in zip(le.classes_, proba):
    print(f"{lbl}: {p:.2%}")